### Задача 0: Подключение и работа с SQLite (0б)

In [1]:
%pip install "sqlalchemy>=2.0" aiosqlite


1. Создайте базу данных `school.db` и создайте таблицу `students` со следующими полями:
   - `id` (INTEGER, PRIMARY KEY, AUTOINCREMENT),
   - `name` (TEXT),
   - `grade` (INTEGER).
2. Вставьте данные о трёх студентах:
   - `Alice`, оценка: 85;
   - `Bob`, оценка: 90;
   - `Charlie`, оценка: 75.
3. Выведите на экран всех студентов с оценкой выше 80.
4. Обновите оценку студента `Charlie` на 80.
5. Удалите студента `Bob`.
6. Выведите всю информацию из базы данных.


In [2]:
import sqlite3

# Подключение к базе данных
connection = sqlite3.connect("school.db")
cursor = connection.cursor()

# Создание таблицы
cursor.execute("""
CREATE TABLE IF NOT EXISTS students (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    grade INTEGER NOT NULL
)
""")

# Вставка данных
cursor.executemany("INSERT INTO students (name, grade) VALUES (?, ?)", [
    ("Alice", 85),
    ("Bob", 90),
    ("Charlie", 75)
])
connection.commit()

# Вывод студентов с оценкой выше 80
cursor.execute("SELECT * FROM students WHERE grade > 80")
print("Студенты с оценкой выше 80:")
for row in cursor.fetchall():
    print(row)

# Обновление оценки Charlie
cursor.execute("UPDATE students SET grade = 80 WHERE name = 'Charlie'")
connection.commit()

# Удаление Bob
cursor.execute("DELETE FROM students WHERE name = 'Bob'")
connection.commit()

# Проверка оставшихся студентов
cursor.execute("SELECT * FROM students")
print("Оставшиеся студенты:")
for row in cursor.fetchall():
    print(row)

cursor.close()
connection.close()

Студенты с оценкой выше 80:
(1, 'Alice', 85)
(2, 'Bob', 90)
Оставшиеся студенты:
(1, 'Alice', 85)
(3, 'Charlie', 80)


### Задача 1: Асинхронная работа с SQLite (1 б)


1. Создайте базу данных `library.db` с таблицей `books`:
   - `id` (INTEGER, PRIMARY KEY, AUTOINCREMENT),
   - `title` (TEXT),
   - `author` (TEXT),
   - `year` (INTEGER).
2. Асинхронно добавьте три книги:
   - `"Book A"`, автор: `Author 1`, год: 2001;
   - `"Book B"`, автор: `Author 2`, год: 1999;
   - `"Book C"`, автор: `Author 3`, год: 2015.
3. Асинхронно получите список всех книг, опубликованных после 2000 года.

In [3]:
import nest_asyncio
import asyncio
import aiosqlite

nest_asyncio.apply()

async def create_database():
    async with aiosqlite.connect("library.db") as connection:
        await connection.execute("""
          CREATE TABLE IF NOT EXISTS books (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            title TEXT NOT NULL,
            author TEXT NOT NULL,
            year INTEGER NOT NULL
          )
        """)

        await connection.commit()
        print('База данных создана')

async def add_books():
    books = [
        ("Book A", "Author 1", 2001),
        ("Book B", "Author 2", 1999),
        ("Book C", "Author 3", 2015)
    ]

    async with aiosqlite.connect("library.db") as connection:
        await connection.execute("DELETE FROM books")
        for title, author, year in books:
            await connection.execute("INSERT INTO books (title, author, year) VALUES (?, ?, ?)", (title, author, year))

        await connection.commit()
        print('Книги добавлены')

async def get_books_after_2000():
    async with aiosqlite.connect("library.db") as connection:
        async with connection.execute("SELECT id, title, author, year FROM books WHERE year > ?", (2000,)) as cursor:
            books = await cursor.fetchall()
            for book in books:
                print(book)

await create_database()
await add_books()
await get_books_after_2000()

База данных создана
Книги добавлены
(1, 'Book A', 'Author 1', 2001)
(3, 'Book C', 'Author 3', 2015)


### Задача 2: Использование оконных функций (1.5б)

1. Создайте SQLite-базу данных `sales.db` с таблицей `sales`:
   - `id` (INTEGER, PRIMARY KEY, AUTOINCREMENT),
   - `region` (TEXT),
   - `employee` (TEXT),
   - `amount` (INTEGER).
2. Вставьте данные:
   - `North`, `Alice`, 500;
   - `North`, `Bob`, 300;
   - `South`, `Charlie`, 700;
   - `South`, `David`, 400;
   - `North`, `Eve`, 200.
3. Напишите SQL-запрос с оконными функциями для:
   - Вычисления общего объёма продаж (`amount`) по каждому региону.
   - Вычисления ранга (`rank`) сотрудника в своём регионе на основе суммы продаж.

**Ожидаемый результат:**
```
('North', 'Alice', 500, 1000, 1)
('North', 'Bob', 300, 1000, 2)
('North', 'Eve', 200, 1000, 3)
('South', 'Charlie', 700, 1100, 1)
('South', 'David', 400, 1100, 2)
```

In [4]:
connection = sqlite3.connect("sales.db")
cursor = connection.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS sales (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    region TEXT NOT NULL,
    employee TEXT NOT NULL,
    amount INTEGER NOT NULL
)
""")

cursor.execute("DELETE FROM sales")
cursor.executemany("INSERT INTO sales (region, employee, amount) VALUES (?, ?, ?)", [
    ("North", "Alice", 500),
    ("North", "Bob", 300),
    ("South", "Charlie", 700),
    ("South", "David", 400),
    ("North", "Eve", 200)
])
connection.commit()

cursor.execute('''
SELECT
    region,
    employee,
    amount,
    SUM(amount) OVER (PARTITION BY region) as region_total,
    RANK() OVER (PARTITION BY region ORDER BY amount DESC) as rank_in_region
FROM sales
ORDER BY region, rank_in_region;
''')
for row in cursor.fetchall():
    print(row)

cursor.close()
connection.close()

('North', 'Alice', 500, 1000, 1)
('North', 'Bob', 300, 1000, 2)
('North', 'Eve', 200, 1000, 3)
('South', 'Charlie', 700, 1100, 1)
('South', 'David', 400, 1100, 2)


### Задача 3: Использование индексов (0.5 б)

1. Создайте SQLite-базу данных `large_library.db` с таблицей `books`:
   - `id` (INTEGER, PRIMARY KEY, AUTOINCREMENT),
   - `title` (TEXT),
   - `author` (TEXT),
   - `year` (INTEGER).
2. Вставьте 1 миллион случайных записей с помощью Python.
3. Создайте индекс на колонке `author`.
4. Напишите запрос для поиска всех книг автора `"Author_100"`. Замерьте время выполнения запроса до и после создания индекса.

In [5]:
import sqlite3
import time
import random

connection = sqlite3.connect("large_library.db")
cursor = connection.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS books (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    author TEXT NOT NULL,
    year INTEGER NOT NULL
)
""")

if not cursor.execute("SELECT COUNT(*) FROM books").fetchone()[0]:
    books = []

    for i in range(1, 1000001):
        title = f"Book_{i}"
        author = f"Author_{random.randint(1, 101)}"
        year = random.randint(2000, 2026)
        books.append((title, author, year))

    cursor.executemany("INSERT INTO books (title, author, year) VALUES (?, ?, ?)", books)
    connection.commit()

start = time.time()
cursor.execute("SELECT * FROM books WHERE author = 'Author_100'")
cursor.fetchall()
print(f"Время выполнения без индекса: {time.time() - start:.4f} секунд")

cursor.execute("CREATE INDEX idx_author ON books(author)")

start = time.time()
cursor.execute("SELECT * FROM books WHERE author = 'Author_100'")
cursor.fetchall()
print(f"Время выполнения с индексом: {time.time() - start:.4f} секунд")

cursor.close()
connection.close()

Время выполнения без индекса: 0.0915 секунд
Время выполнения с индексом: 0.0299 секунд


### Задача 4: Использование ограничений (constraints)  (0.5 б)

1. Создайте базу данных `university.db` с таблицей `students`:
   - `id` (INTEGER, PRIMARY KEY, AUTOINCREMENT),
   - `name` (TEXT, NOT NULL),
   - `email` (TEXT, UNIQUE),
   - `age` (INTEGER, CHECK(age >= 18)).
2. Попробуйте вставить записи:
   - `Alice`, `alice@example.com`, 20;
   - `Bob`, `bob@example.com`, 17 (должна вызвать ошибку CHECK);
   - `Charlie`, `alice@example.com`, 22 (должна вызвать ошибку UNIQUE).
3. Добавьте индекс на поле `name` для ускорения поиска.
4. Напишите запрос для выборки студентов, чей возраст больше 19.

In [6]:
import sqlite3

connection = sqlite3.connect("university.db")
cursor = connection.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS students (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    email TEXT UNIQUE,
    age INTEGER CHECK(age >= 18)
)
""")

for row in [
    ("Alice", "alice@example.com", 20),
    ("Bob", "bob@example.com", 17),       # Ошибка CHECK
    ("Charlie", "alice@example.com", 22), # Ошибка UNIQUE
]:
    try:
        cursor.execute("INSERT INTO students (name, email, age) VALUES (?, ?, ?)", row)
    except sqlite3.IntegrityError as e:
        print(f"Ошибка вставки для {row[0]}: {e}")

connection.commit()

cursor.execute("CREATE INDEX IF NOT EXISTS idx_name ON students(name)")

cursor.execute("SELECT * FROM students WHERE age > 19")
for row in cursor.fetchall():
    print(row)

cursor.close()
connection.close()

Ошибка вставки для Bob: CHECK constraint failed: age >= 18
Ошибка вставки для Charlie: UNIQUE constraint failed: students.email
(1, 'Alice', 'alice@example.com', 20)


### Задача 2026: Создание базовой ORM-модели и работа с данными (0 б)


1. Создайте базу данных `company.db` с таблицами:
   - `departments`:
     - `id` (INTEGER, PRIMARY KEY, AUTOINCREMENT),
     - `name` (TEXT, UNIQUE, NOT NULL).
   - `employees`:
     - `id` (INTEGER, PRIMARY KEY, AUTOINCREMENT),
     - `name` (TEXT, NOT NULL),
     - `salary` (INTEGER, CHECK(salary > 0)),
     - `department_id` (INTEGER, ForeignKey(`departments.id`)).
2. Добавьте 3 отдела:
   - `HR`, `IT`, `Sales`.
3. Добавьте по 2 сотрудника в каждый отдел.
4. Напишите запросы:
   - Вывести всех сотрудников с их отделами.
   - Увеличить зарплату сотрудников отдела `IT` на 10%.
   - Удалить сотрудников из отдела `Sales`.


In [7]:
from __future__ import annotations

from typing import List

from sqlalchemy import CheckConstraint, ForeignKey, String, create_engine, delete, select, update
from sqlalchemy.orm import DeclarativeBase, Mapped, Session, mapped_column, relationship, joinedload

# Модели
class Base(DeclarativeBase):
    pass


class Department(Base):
    __tablename__ = "departments"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String, unique=True, nullable=False)

    employees: Mapped[List["Employee"]] = relationship(
        back_populates="department",
        cascade="all, delete-orphan",
    )


class Employee(Base):
    __tablename__ = "employees"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String, nullable=False)
    salary: Mapped[int] = mapped_column(
        CheckConstraint("salary > 0"),
        nullable=False,
    )
    department_id: Mapped[int] = mapped_column(ForeignKey("departments.id"))
    department: Mapped[Department] = relationship(back_populates="employees")

# Создание базы данных
engine = create_engine("sqlite:///company.db")
Base.metadata.create_all(engine)

with Session(engine) as session:
    has_any_department = session.scalar(select(Department.id).limit(1)) is not None
    # Добавление данных
    if not has_any_department:
        hr = Department(
            name="HR",
            employees=[
                Employee(name="Alice", salary=5000),
                Employee(name="Bob", salary=4500),
            ],
        )
        it = Department(
            name="IT",
            employees=[
                Employee(name="Charlie", salary=6000),
                Employee(name="David", salary=7000),
            ],
        )
        sales = Department(
            name="Sales",
            employees=[
                Employee(name="Eve", salary=4000),
                Employee(name="Frank", salary=3000),
            ],
        )
        session.add_all([hr, it, sales])
        session.commit()
    # Запросы
    # 1. Вывести всех сотрудников с отделами
    employees = session.scalars(
        select(Employee).options(joinedload(Employee.department))
    ).all()

    print("Все сотрудники с отделами:")
    for employee in employees:
        print(f"{employee.name} - {employee.department.name}, зарплата: {employee.salary}")

    # 2. Увеличить зарплату сотрудников IT на 10%
    session.execute(
        update(Employee)
        .where(Employee.department.has(Department.name == "IT"))
        .values(salary=Employee.salary * 1.1)
    )
    session.commit()

    # 3. Удалить сотрудников из отдела Sales
    sales_id = session.scalar(select(Department.id).where(Department.name == "Sales"))
    if sales_id is not None:
        session.execute(delete(Employee).where(Employee.department_id == sales_id))
        session.commit()
    # Проверка оставшихся сотрудников
    print("Сотрудники после изменений:")
    for employee in session.scalars(
        select(Employee).options(joinedload(Employee.department))
    ):
        print(f"{employee.name} - {employee.department.name}, зарплата: {employee.salary}")

Все сотрудники с отделами:
Alice - HR, зарплата: 5000
Bob - HR, зарплата: 4500
Charlie - IT, зарплата: 6000
David - IT, зарплата: 7000
Eve - Sales, зарплата: 4000
Frank - Sales, зарплата: 3000
Сотрудники после изменений:
Alice - HR, зарплата: 5000
Bob - HR, зарплата: 4500
Charlie - IT, зарплата: 6600.000000000001
David - IT, зарплата: 7700.000000000001


### Задача 5: Работа с ORM: связь многие-ко-многим (1.5б)


1. Создайте базу данных `school.db` с таблицами:
   - `students`:
     - `id` (INTEGER, PRIMARY KEY, AUTOINCREMENT),
     - `name` (TEXT, NOT NULL).
   - `courses`:
     - `id` (INTEGER, PRIMARY KEY, AUTOINCREMENT),
     - `title` (TEXT, UNIQUE, NOT NULL).
   - Связующая таблица `student_courses`:
     - `student_id` (INTEGER, ForeignKey(`students.id`)),
     - `course_id` (INTEGER, ForeignKey(`courses.id`)).
2. Добавьте студентов и курсы:
   - Студенты: `Alice`, `Bob`, `Charlie`.
   - Курсы: `Math`, `Physics`, `Chemistry`.
3. Запишите данные о том, кто посещает какие курсы:
   - `Alice` посещает `Math` и `Physics`.
   - `Bob` посещает `Physics` и `Chemistry`.
   - `Charlie` посещает все три курса.
4. Напишите запросы:
   - Вывести всех студентов с их курсами.
   - Найти всех студентов, которые посещают `Physics`.

**Ожидаемый вывод**
```
Alice: Physics, Math
Charlie: Physics, Chemistry, Math
Bob: Physics, Chemistry
Студенты, посещающие Physics:
Alice
Bob
Charlie
```

In [8]:
from __future__ import annotations

from typing import List

from sqlalchemy import CheckConstraint, ForeignKey, String, create_engine, delete, select, update
from sqlalchemy.orm import DeclarativeBase, Mapped, Session, mapped_column, relationship, joinedload

class Base(DeclarativeBase):
    pass


class StudentCourse(Base):
    __tablename__ = "student_courses"

    student_id: Mapped[int] = mapped_column(ForeignKey("students.id"), primary_key = True)
    course_id: Mapped[int] = mapped_column(ForeignKey("courses.id"), primary_key = True)


class Student(Base):
    __tablename__ = "students"

    id: Mapped[int] = mapped_column(primary_key = True)
    name: Mapped[str] = mapped_column(String, nullable = False)
    student_courses: Mapped[List["StudentCourse"]] = relationship(back_populates = "student")


class Course(Base):
    __tablename__ = "courses"

    id: Mapped[int] = mapped_column(primary_key = True)
    title: Mapped[str] = mapped_column(String, unique = True, nullable = False)
    course_students: Mapped[List["StudentCourse"]] = relationship(back_populates = "course")


StudentCourse.student = relationship(Student, back_populates = "student_courses")
StudentCourse.course = relationship(Course, back_populates = "course_students")

engine = create_engine("sqlite:///school.db")
Base.metadata.drop_all(engine)
Base.metadata.create_all(engine)

with Session(engine) as session:
    math = Course(title = "Math")
    physics = Course(title = "Physics")
    chemistry = Course(title = "Chemistry")
    session.add_all([math, physics, chemistry])
    session.flush()

    alice = Student(name = "Alice")
    bob = Student(name = "Bob")
    charlie = Student(name = "Charlie")
    session.add_all([alice, bob, charlie])
    session.flush()

    session.add_all([
        StudentCourse(student = alice, course = math),
        StudentCourse(student = alice, course = physics),
        StudentCourse(student = bob, course = physics),
        StudentCourse(student = bob, course = chemistry),
        StudentCourse(student = charlie, course = math),
        StudentCourse(student = charlie, course = physics),
        StudentCourse(student = charlie, course = chemistry),
    ])
    session.commit()

print("Все студенты с их курсами:")
students = session.query(Student).all()
for student in students:
    courses = [sc.course.title for sc in student.student_courses]
    print(f"{student.name}: {', '.join(sorted(courses))}")

print("Студенты, которые посещают Physics:")
physics_students = session.query(Student).join(StudentCourse).join(Course).filter(Course.title == "Physics").all()
for student in physics_students:
    print(student.name)

Все студенты с их курсами:
Alice: Math, Physics
Bob: Chemistry, Physics
Charlie: Chemistry, Math, Physics
Студенты, которые посещают Physics:
Alice
Bob
Charlie


### Задача 6: Оптимизация запросов с использованием `joinedload` из ORM (0.5 б)

1. Создайте SQLite-базу данных `social.db` с таблицами:
   - `users`: `id` (INTEGER, PRIMARY KEY), `name` (TEXT);
   - `posts`: `id` (INTEGER, PRIMARY KEY), `title` (TEXT), `user_id` (INTEGER).
2. Заполните таблицы следующими данными:
   - Пользователи: `Alice`, `Bob`.
   - Посты: для `Alice` — `Post 1`, `Post 2`; для `Bob` — `Post 3`.
3. Напишите код, который эффективно выводит пользователей и их посты с использованием `joinedload`.

In [9]:
from __future__ import annotations

from typing import List

from sqlalchemy import ForeignKey, String, create_engine, select
from sqlalchemy.orm import DeclarativeBase, Mapped, Session, mapped_column, relationship, joinedload

class Base(DeclarativeBase):
    pass


class User(Base):
    __tablename__ = "users"

    id: Mapped[int] = mapped_column(primary_key = True)
    name: Mapped[str] = mapped_column(String)

    posts: Mapped[List["Post"]] = relationship(back_populates = "user")


class Post(Base):
    __tablename__ = "posts"

    id: Mapped[int] = mapped_column(primary_key = True)
    title: Mapped[str] = mapped_column(String)
    user_id: Mapped[int] = mapped_column(ForeignKey("users.id"))

    user: Mapped[User] = relationship(back_populates = "posts")

engine = create_engine("sqlite:///social.db")
Base.metadata.create_all(engine)

with Session(engine) as session:
    alice = User(name = "Alice")
    bob = User(name = "Bob")

    post1 = Post(title = "Post 1", user = alice)
    post2 = Post(title = "Post 2", user = alice)
    post3 = Post(title = "Post 3", user = bob)

    session.add_all([alice, bob, post1, post2, post3])
    session.commit()

    stmt = select(User).options(joinedload(User.posts))
    users = session.scalars(stmt).unique().all()

    for user in users:
        post_titles = [post.title for post in user.posts]
        print(f"{user.name}: {', '.join(post_titles)}")

Alice: Post 1, Post 2
Bob: Post 3


### Задача 7: Оконные функции в ORM (2.5б)

1. Создайте базу данных `sales.db` с таблицей `sales`:
   - `id` (INTEGER, PRIMARY KEY, AUTOINCREMENT),
   - `region` (TEXT),
   - `employee` (TEXT),
   - `amount` (INTEGER).
2. Вставьте данные:
   - `North`, `Alice`, 500;
   - `North`, `Bob`, 300;
   - `South`, `Charlie`, 700;
   - `South`, `David`, 400;
   - `North`, `Eve`, 200.
3. Используйте оконные функции для:
   - Подсчёта общего объёма продаж в каждом регионе.
   - Определения ранга сотрудника по продажам в своём регионе.

**Ожидаемый вывод**
```
('North', 'Alice', 500, 1000, 1)
('North', 'Bob', 300, 1000, 2)
('North', 'Eve', 200, 1000, 3)
('South', 'Charlie', 700, 1100, 1)
('South', 'David', 400, 1100, 2)
```

In [10]:
from __future__ import annotations

from typing import List

from sqlalchemy import ForeignKey, String, create_engine, select, func
from sqlalchemy.orm import DeclarativeBase, Mapped, Session, mapped_column, relationship, joinedload

class Base(DeclarativeBase):
    pass


class Sale(Base):
    __tablename__ = "sales"

    id: Mapped[int] = mapped_column(primary_key = True)
    region: Mapped[str] = mapped_column(String)
    employee: Mapped[str] = mapped_column(String)
    amount: Mapped[int] = mapped_column()


engine = create_engine("sqlite:///sales.db")
Base.metadata.drop_all(engine)
Base.metadata.create_all(engine)

with Session(engine) as session:
    data = [
        Sale(region = "North", employee = "Alice", amount = 500),
        Sale(region = "North", employee = "Bob", amount = 300),
        Sale(region = "South", employee = "Charlie", amount = 700),
        Sale(region = "South", employee = "David", amount = 400),
        Sale(region = "North", employee = "Eve", amount = 200)
    ]
    session.add_all(data)
    session.commit()

    stmt = select(
        Sale.region,
        Sale.employee,
        Sale.amount,
        func.sum(Sale.amount).over(partition_by = Sale.region).label("total_region"),
        func.rank().over(partition_by = Sale.region, order_by = Sale.amount.desc()).label("rank")
    ).order_by(Sale.region, func.rank().over(partition_by = Sale.region, order_by = Sale.amount.desc()))

for row in session.execute(stmt).all():
    print(row)

('North', 'Alice', 500, 1000, 1)
('North', 'Bob', 300, 1000, 2)
('North', 'Eve', 200, 1000, 3)
('South', 'Charlie', 700, 1100, 1)
('South', 'David', 400, 1100, 2)
